Transaction, Concurrency & Recovery (ภาคปฏิบัติ)
**ฐานข้อมูลขั้นสูง · MySQL 8.4 / InnoDB · Python + mysql-connector-python**

Notebook นี้รวมโค้ดภาคปฏิบัติจากเอกสารหน้า 20–31 (หัวข้อ 2.1–2.5) ให้รันได้ต่อเนื่องจากบนลงล่าง

| ส่วน | เนื้อหา |
|---|---|
| 2.1 | เตรียมสภาพแวดล้อม, สร้าง database, schema, ข้อมูลตัวอย่าง |
| 2.2 | Commit/Rollback, Create Post + Metric, Deadlock Retry |
| 2.3 | Lost Update, Atomic UPDATE, Deadlock, ทดสอบ Isolation Level |
| 2.4 | InnoDB Status, Redo Log configuration, วัด redo bytes |
| 2.5 | Lab 1–4 (Dirty Read, Report Fake News, Benchmark, Deadlock Retry) |

> **ผลของ concurrency (race condition / deadlock) เป็น nondeterministic** — ตัวเลขแต่ละรอบอาจไม่เท่ากัน ให้อธิบายเหตุผลจากผลที่ได้

## 2.1 การเตรียมสภาพแวดล้อม

### 2.1.1 ติดตั้ง Python Dependencies

In [ ]:
!pip -q install mysql-connector-python pandas

import mysql.connector
from mysql.connector import Error
import pandas as pd
print("mysql-connector-python", mysql.connector.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 49.3 MB/s eta 0:00:00
mysql-connector-python 26.7.0


### เลือกแหล่ง MySQL Server

มี 2 ทางเลือก ให้ตั้งค่า `USE_LOCAL_COLAB_MYSQL` ในเซลล์ถัดไป

* **`False` (แนะนำตามเอกสาร)** — ต่อไปยัง MySQL 8.4 ที่ผู้สอนเตรียมไว้ (server มหาวิทยาลัย / VM / managed DB) ระบบจะถาม password ด้วย `getpass` เพื่อไม่ให้ password จริงติดอยู่ใน Notebook
* **`True` (สำรอง ใช้ฝึกเอง)** — ติดตั้ง MySQL ลงในเครื่อง Colab ชั่วคราวด้วย `apt`
  ⚠️ แพ็กเกจของ Ubuntu เป็น **MySQL 8.0** ไม่ใช่ 8.4 แนวคิดและคำสั่งในแลปนี้ใช้ได้เหมือนกัน แต่ในรายงานต้องระบุเวอร์ชันที่ใช้จริง (ดู `SELECT VERSION()`)

In [ ]:
USE_LOCAL_COLAB_MYSQL = True   # เปลี่ยนเป็น False ถ้าต่อ server ของผู้สอน

import os, shutil, subprocess, getpass

if USE_LOCAL_COLAB_MYSQL:
    LAB_USER, LAB_PASSWORD = 'lab', 'LabPass#2026'   # ใช้เฉพาะ server ชั่วคราวใน Colab เท่านั้น
    if shutil.which('mysqld') is None:
        print("กำลังติดตั้ง MySQL server (ประมาณ 1-2 นาที)...")
        subprocess.run("apt-get -qq update > /dev/null && "
                       "DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null",
                       shell=True, check=True)
    subprocess.run("service mysql start > /dev/null 2>&1", shell=True)
    # สร้างบัญชี lab ด้วย authentication ค่าเริ่มต้นของ server (caching_sha2_password) — ไม่ใช้ mysql_native_password
    subprocess.run(["mysql", "-e",
        f"CREATE USER IF NOT EXISTS '{LAB_USER}'@'localhost' IDENTIFIED BY '{LAB_PASSWORD}';"
        f"GRANT ALL PRIVILEGES ON *.* TO '{LAB_USER}'@'localhost'; FLUSH PRIVILEGES;"], check=True)
    SERVER_CONFIG = {'host': '127.0.0.1', 'port': 3306,
                     'user': LAB_USER, 'password': LAB_PASSWORD, 'connection_timeout': 10}
else:
    SERVER_CONFIG = {
        'host': 'YOUR_HOST',          # แก้เป็น host ที่ผู้สอนให้
        'port': 3306,
        'user': 'YOUR_USER',
        'password': getpass.getpass('MySQL password: '),
        'connection_timeout': 10,
    }
print("SERVER_CONFIG พร้อม:", {k: v for k, v in SERVER_CONFIG.items() if k != 'password'})

กำลังติดตั้ง MySQL server (ประมาณ 1-2 นาที)...
SERVER_CONFIG พร้อม: {'host': '127.0.0.1', 'port': 3306, 'user': 'lab', 'connection_timeout': 10}


### 2.1.2 ตรวจสอบ MySQL Server

In [ ]:
conn = mysql.connector.connect(**SERVER_CONFIG)
cur = conn.cursor()
cur.execute("SELECT VERSION(), @@version_comment, @@transaction_isolation, @@autocommit")
version, comment, iso, ac = cur.fetchone()
print(f"Server version : {version} ({comment})")
print(f"Default isolation: {iso} | autocommit: {ac}")
if 'mariadb' in comment.lower():
    print("⚠️ server นี้เป็น MariaDB ไม่ใช่ MySQL — พฤติกรรมบางส่วนจะต่างจากเอกสาร")
elif not version.startswith('8.4'):
    print("ℹ️ ไม่ใช่ MySQL 8.4 — ให้ระบุเวอร์ชันนี้ในรายงาน")
cur.close(); conn.close()

Server version : 8.0.46-0ubuntu0.24.04.4 ((Ubuntu))
Default isolation: REPEATABLE-READ | autocommit: 0
ℹ️ ไม่ใช่ MySQL 8.4 — ให้ระบุเวอร์ชันนี้ในรายงาน


### 2.1.3 สร้าง Database ก่อน แล้วจึงเชื่อมด้วย DB_CONFIG
เชื่อมระดับ server ก่อน (ยังไม่ระบุ database) → `CREATE DATABASE` → จากนั้นจึงสร้าง `DB_CONFIG` ที่ระบุ database

In [ ]:
DB_NAME = 'fake_news_db'   # ถ้าใช้ server ร่วมกันหลายกลุ่ม ให้เปลี่ยนเป็น fake_news_db_กลุ่มX

# Step 1: connect ระดับ server
conn = mysql.connector.connect(**SERVER_CONFIG)
cur = conn.cursor()
cur.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} CHARACTER SET utf8mb4")
cur.close(); conn.close()

# Step 2: config ที่ระบุ database
DB_CONFIG = {**SERVER_CONFIG, 'database': DB_NAME, 'autocommit': False}

def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

def query_df(sql, params=None):
    # helper: รัน SELECT แล้วคืนผลเป็น DataFrame (ใช้แสดงผลใน notebook)
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute(sql, params or ())
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)
    finally:
        cur.close(); conn.close()

print("เชื่อมต่อ database:", query_df("SELECT DATABASE() AS db").iloc[0, 0])

เชื่อมต่อ database: fake_news_db


### 2.1.4 สร้าง Schema สำหรับ Fake News Mini Project

In [ ]:
def setup_database():
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute('''
            CREATE TABLE IF NOT EXISTS users (
                user_id     VARCHAR(20) PRIMARY KEY,
                name        VARCHAR(100) NOT NULL,
                email       VARCHAR(100) UNIQUE,
                followers   INT NOT NULL DEFAULT 0,
                post_count  INT NOT NULL DEFAULT 0,
                is_verified BOOLEAN NOT NULL DEFAULT FALSE,
                created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                CHECK (followers >= 0),
                CHECK (post_count >= 0)
            ) ENGINE=InnoDB
        ''')
        cur.execute('''
            CREATE TABLE IF NOT EXISTS posts (
                post_id      VARCHAR(20) PRIMARY KEY,
                author_id    VARCHAR(20) NOT NULL,
                content      TEXT NOT NULL,
                share_count  INT NOT NULL DEFAULT 0,
                report_count INT NOT NULL DEFAULT 0,
                is_flagged   BOOLEAN NOT NULL DEFAULT FALSE,
                created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                CONSTRAINT fk_posts_author
                    FOREIGN KEY (author_id) REFERENCES users(user_id)
                    ON DELETE CASCADE,
                CHECK (share_count >= 0),
                CHECK (report_count >= 0)
            ) ENGINE=InnoDB
        ''')
        cur.execute('''
            CREATE TABLE IF NOT EXISTS shares (
                share_id  BIGINT AUTO_INCREMENT PRIMARY KEY,
                post_id   VARCHAR(20) NOT NULL,
                user_id   VARCHAR(20) NOT NULL,
                shared_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                CONSTRAINT fk_shares_post FOREIGN KEY (post_id) REFERENCES posts(post_id) ON DELETE CASCADE,
                CONSTRAINT fk_shares_user FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE,
                CONSTRAINT uq_share UNIQUE (post_id, user_id)
            ) ENGINE=InnoDB
        ''')
        conn.commit()
    except Error:
        conn.rollback()
        raise
    finally:
        cur.close(); conn.close()

setup_database()
query_df("SHOW TABLES")

,Tables_in_fake_news_db
0,posts
1,shares
2,users


### 2.1.5 ใส่ข้อมูลตัวอย่างแบบทำซ้ำได้ (Idempotent-ish Reset)
เรียก `insert_sample_data()` ซ้ำได้ทุกครั้งที่ต้องการรีเซ็ตข้อมูลกลับเป็นค่าเริ่มต้น

In [ ]:
def insert_sample_data():
    conn = get_connection(); cur = conn.cursor()
    try:
        # ลบตาม dependency: ตารางลูกก่อน แล้วจึงตารางแม่
        for t in ('reports', 'shares', 'posts', 'users'):
            cur.execute(f"SELECT COUNT(*) FROM information_schema.tables "
                        f"WHERE table_schema=DATABASE() AND table_name='{t}'")
            if cur.fetchone()[0]:
                cur.execute(f"DELETE FROM {t}")
        users = [
            ('U001', 'Alice',   'alice@test.com',   1500, 0, True),
            ('U002', 'Bob',     'bob@test.com',      300, 0, False),
            ('U003', 'Charlie', 'charlie@test.com',  800, 0, True),
            ('U004', 'David',   'david@test.com',    120, 0, False),
        ]
        cur.executemany('''
            INSERT INTO users(user_id, name, email, followers, post_count, is_verified)
            VALUES (%s, %s, %s, %s, %s, %s)
        ''', users)
        posts = [
            ('P001', 'U001', 'Breaking: New AI model released!', 0, 0, False),
            ('P002', 'U002', 'Is this AI news fake?',            0, 0, False),
            ('P003', 'U003', 'Government announces new policy',  0, 0, False),
        ]
        cur.executemany('''
            INSERT INTO posts(post_id, author_id, content, share_count, report_count, is_flagged)
            VALUES (%s, %s, %s, %s, %s, %s)
        ''', posts)
        conn.commit()
    except Error:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

insert_sample_data()
display(query_df("SELECT user_id, name, followers, post_count FROM users"))
display(query_df("SELECT post_id, author_id, content, share_count, report_count, is_flagged FROM posts"))

,user_id,name,followers,post_count
0,U001,Alice,1500,0
1,U002,Bob,300,0
2,U003,Charlie,800,0
3,U004,David,120,0


,post_id,author_id,content,share_count,report_count,is_flagged
0,P001,U001,Breaking: New AI model released!,0,0,0
1,P002,U002,Is this AI news fake?,0,0,0
2,P003,U003,Government announces new policy,0,0,0


## 2.2 การจัดการ Transaction ใน MySQL ผ่าน Python

### 2.2.1 รูปแบบ Commit/Rollback ที่แนะนำ — พร้อมพิสูจน์ (งาน A)
ทดลอง 2 กรณีกับ business operation “โอน followers + บันทึก share” ที่ statement ที่ 2 ผิด (duplicate key ใน `uq_share`)

1. **ไม่ rollback แต่ commit ต่อ** → statement แรกที่สำเร็จแล้วยังถูก commit (InnoDB rollback แค่ statement ที่ผิด)
2. **ใช้ try/except + rollback** → ทั้ง business operation ถูกยกเลิก (atomic)

In [ ]:
def followers_of(uid):
    return int(query_df("SELECT followers FROM users WHERE user_id=%s", (uid,)).iloc[0, 0])

# เตรียม: ให้มี share (P001, U002) อยู่แล้ว เพื่อให้ INSERT ซ้ำเกิด duplicate key
insert_sample_data()
conn = get_connection(); cur = conn.cursor()
cur.execute("INSERT INTO shares(post_id, user_id) VALUES ('P001', 'U002')"); conn.commit()
cur.close(); conn.close()

# ---- กรณีที่ 1: จับ error แต่ไม่ rollback แล้ว commit ----
before = followers_of('U001')
conn = get_connection(); cur = conn.cursor()
try:
    cur.execute("UPDATE users SET followers = followers + 100 WHERE user_id='U001'")    # สำเร็จ
    cur.execute("INSERT INTO shares(post_id, user_id) VALUES ('P001', 'U002')")        # duplicate!
except Error as e:
    print(f"[กรณี 1] error {e.errno}: {e.msg}")
    conn.commit()   # ❌ ผิดแบบ: commit ทั้งที่งานยังไม่ครบ
finally:
    cur.close(); conn.close()
print(f"[กรณี 1] followers U001: {before} -> {followers_of('U001')}  (UPDATE แรกยังอยู่ = ไม่ atomic)\n")

# ---- กรณีที่ 2: รูปแบบที่แนะนำ ----
before = followers_of('U001')
conn = get_connection(); cur = conn.cursor()
try:
    cur.execute("UPDATE users SET followers = followers + 100 WHERE user_id='U001'")
    cur.execute("INSERT INTO shares(post_id, user_id) VALUES ('P001', 'U002')")
    conn.commit()
except Error as e:
    conn.rollback()  # ✅ ย้อนทั้ง business operation
    print(f"[กรณี 2] error {e.errno}: {e.msg} -> rollback ทั้ง transaction")
finally:
    cur.close(); conn.close()
print(f"[กรณี 2] followers U001: {before} -> {followers_of('U001')}  (ไม่เปลี่ยน = atomic)")

[กรณี 1] error 1062: Duplicate entry 'P001-U002' for key 'shares.uq_share'
[กรณี 1] followers U001: 1500 -> 1600  (UPDATE แรกยังอยู่ = ไม่ atomic)

[กรณี 2] error 1062: Duplicate entry 'P001-U002' for key 'shares.uq_share' -> rollback ทั้ง transaction
[กรณี 2] followers U001: 1600 -> 1600  (ไม่เปลี่ยน = atomic)


### 2.2.2 ตัวอย่าง: Create Post + Update Metric
ตรวจ `rowcount` เพราะ UPDATE ที่ไม่ match แถวใด **ไม่ใช่ database error** — ต้อง validate เอง

In [ ]:
import uuid

def create_post_with_metrics(user_id, content):
    post_id = f"P{uuid.uuid4().hex[:8]}"
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute('''
            INSERT INTO posts(post_id, author_id, content, share_count, report_count)
            VALUES (%s, %s, %s, 0, 0)
        ''', (post_id, user_id, content))
        cur.execute('''
            UPDATE users SET post_count = post_count + 1
            WHERE user_id = %s
        ''', (user_id,))
        if cur.rowcount != 1:
            raise ValueError(f"Unknown user_id: {user_id}")
        conn.commit()
        return post_id
    except Exception:
        conn.rollback()
        raise
    finally:
        cur.close(); conn.close()

insert_sample_data()
new_id = create_post_with_metrics('U001', 'AI ช่วยตรวจข่าวปลอมได้จริงไหม?')
print("สร้างโพสต์สำเร็จ:", new_id)

try:
    create_post_with_metrics('U999', 'โพสต์จาก user ที่ไม่มีอยู่')
except Exception as e:
    print(f"ล้มเหลวตามคาด -> {type(e).__name__}: {e}")

display(query_df("SELECT user_id, post_count FROM users ORDER BY user_id"))
display(query_df("SELECT post_id, author_id, content FROM posts ORDER BY created_at, post_id"))

สร้างโพสต์สำเร็จ: Pe242543a
ล้มเหลวตามคาด -> IntegrityError: 1452 (23000): Cannot add or update a child row: a foreign key constraint fails (`fake_news_db`.`posts`, CONSTRAINT `fk_posts_author` FOREIGN KEY (`author_id`) REFERENCES `users` (`user_id`) ON DELETE CASCADE)


,user_id,post_count
0,U001,1
1,U002,0
2,U003,0
3,U004,0


,post_id,author_id,content
0,P001,U001,Breaking: New AI model released!
1,P002,U002,Is this AI news fake?
2,P003,U003,Government announces new policy
3,Pe242543a,U001,AI ช่วยตรวจข่าวปลอมได้จริงไหม?


### 2.2.3 Deadlock Retry
* **1213 Deadlock** — InnoDB rollback *ทั้ง transaction* ของ victim แล้ว → ต้อง retry ทั้ง transaction
* **1205 Lock wait timeout** — ค่า default rollback เฉพาะ statement (เว้นแต่ตั้ง `innodb_rollback_on_timeout`) ฟังก์ชัน `fn` จึงต้อง rollback เองก่อน raise ซึ่งทุกฟังก์ชันใน notebook นี้ทำอยู่แล้ว

In [ ]:
import time, random

DEADLOCK = 1213
LOCK_WAIT_TIMEOUT = 1205

def run_with_retry(fn, max_retries=5, verbose=True):
    for attempt in range(max_retries):
        try:
            return fn()
        except Error as e:
            if e.errno not in (DEADLOCK, LOCK_WAIT_TIMEOUT) or attempt == max_retries - 1:
                raise
            # exponential backoff + jitter
            wait = 0.1 * (2 ** attempt) + random.uniform(0, 0.05)
            if verbose:
                print(f"  ↻ errno {e.errno} (attempt {attempt + 1}) -> retry อีกใน {wait:.2f}s")
            time.sleep(wait)

## 2.3 การจำลอง Race Condition

### 2.3.1 Unsafe Read-Modify-Write: Lost Update (งาน B)
แต่ละ bot ทำงานใน transaction ของตัวเอง แต่ **อ่านค่า → คำนวณใน Python → เขียนกลับ** แยกเป็นหลาย statement โดยไม่มี lock/version check จึงเกิด lost update

In [ ]:
import concurrent.futures

def bot_share_unsafe(bot_id, post_id):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute("SELECT share_count FROM posts WHERE post_id=%s", (post_id,))
        current = cur.fetchone()[0]
        time.sleep(0.02)  # เพิ่มโอกาส race
        cur.execute("UPDATE posts SET share_count=%s WHERE post_id=%s", (current + 1, post_id))
        conn.commit()
        return current + 1
    except Exception:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

def reset_share(post_id='P001'):
    with get_connection() as conn:
        cur = conn.cursor()
        cur.execute("UPDATE posts SET share_count=0 WHERE post_id=%s", (post_id,))
        conn.commit(); cur.close()

def get_share(post_id='P001'):
    return int(query_df("SELECT share_count FROM posts WHERE post_id=%s", (post_id,)).iloc[0, 0])

N_BOTS = 10
reset_share()
with concurrent.futures.ThreadPoolExecutor(max_workers=N_BOTS) as ex:
    returned = list(ex.map(lambda i: bot_share_unsafe(i, 'P001'), range(N_BOTS)))

unsafe_result = get_share()
print("ค่าที่แต่ละ bot เขียน:", sorted(returned))
print(f"คาดหวัง {N_BOTS} | ได้จริง {unsafe_result} | lost updates = {N_BOTS - unsafe_result}")

ค่าที่แต่ละ bot เขียน: [1, 2, 3, 3, 3, 3, 3, 4, 4, 4]
คาดหวัง 10 | ได้จริง 4 | lost updates = 6


### 2.3.2 แก้ด้วย Atomic UPDATE

In [ ]:
def bot_share_atomic(bot_id, post_id):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute('''
            UPDATE posts
            SET share_count = share_count + 1
            WHERE post_id = %s
        ''', (post_id,))
        conn.commit()
    except Exception:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

reset_share()
with concurrent.futures.ThreadPoolExecutor(max_workers=N_BOTS) as ex:
    list(ex.map(lambda i: bot_share_atomic(i, 'P001'), range(N_BOTS)))
atomic_result = get_share()
print(f"Atomic UPDATE: คาดหวัง {N_BOTS} | ได้จริง {atomic_result}")

Atomic UPDATE: คาดหวัง 10 | ได้จริง 10


#### (เสริม) แก้ด้วย `SELECT ... FOR UPDATE`
กรณีจำเป็นต้องอ่านค่ามาคำนวณใน application ก่อนเขียน ให้ใช้ locking read เพื่อถือ X lock ไว้จน commit

In [ ]:
def bot_share_for_update(bot_id, post_id):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute("SELECT share_count FROM posts WHERE post_id=%s FOR UPDATE", (post_id,))
        current = cur.fetchone()[0]
        time.sleep(0.02)
        cur.execute("UPDATE posts SET share_count=%s WHERE post_id=%s", (current + 1, post_id))
        conn.commit()
    except Exception:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

reset_share()
with concurrent.futures.ThreadPoolExecutor(max_workers=N_BOTS) as ex:
    list(ex.map(lambda i: bot_share_for_update(i, 'P001'), range(N_BOTS)))
locking_result = get_share()
print(f"SELECT ... FOR UPDATE: คาดหวัง {N_BOTS} | ได้จริง {locking_result}")

pd.DataFrame({'วิธี': ['Unsafe read-modify-write', 'Atomic UPDATE', 'SELECT ... FOR UPDATE'],
              'share_count': [unsafe_result, atomic_result, locking_result],
              'คาดหวัง': [N_BOTS] * 3})

SELECT ... FOR UPDATE: คาดหวัง 10 | ได้จริง 10


,วิธี,share_count,คาดหวัง
0,Unsafe read-modify-write,4,10
1,Atomic UPDATE,10,10
2,SELECT ... FOR UPDATE,10,10


### 2.3.3 จำลอง Deadlock จริงด้วย Row Locks (งาน D)
Thread 1 โอน U001 → U002 และ Thread 2 โอน U002 → U001 พร้อมกัน ลำดับ lock ตรงข้ามกัน
ใช้ `threading.Barrier` บังคับให้ทั้งสองถือ lock แถวแรกก่อน แล้วค่อยขอแถวที่สอง จึงเกิด deadlock แทบทุกครั้ง (ฟังก์ชันตามเอกสารใช้ `sleep(0.05)` ซึ่งอาจเกิดหรือไม่เกิดก็ได้)

In [ ]:
import threading

def transfer_followers(from_user, to_user, amount, barrier=None):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute("UPDATE users SET followers=followers-%s WHERE user_id=%s", (amount, from_user))
        if barrier is not None:
            try:
                barrier.wait(timeout=5)   # ใช้แค่รอบแรก (retry จะไม่รอ barrier ซ้ำ)
            except threading.BrokenBarrierError:
                pass
        else:
            time.sleep(0.05)
        cur.execute("UPDATE users SET followers=followers+%s WHERE user_id=%s", (amount, to_user))
        conn.commit()
        return 'committed'
    except Error:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

def run_two_transfers(use_retry):
    insert_sample_data()
    barrier = threading.Barrier(2)
    results = {}
    def job(name, a, b):
        first = [True]
        def once():
            bar = barrier if first[0] else None
            first[0] = False
            return transfer_followers(a, b, 10, bar)
        try:
            results[name] = run_with_retry(once) if use_retry else once()
        except Error as e:
            results[name] = f"error {e.errno}: {e.msg}"
    t1 = threading.Thread(target=job, args=('T1 U001->U002', 'U001', 'U002'))
    t2 = threading.Thread(target=job, args=('T2 U002->U001', 'U002', 'U001'))
    t1.start(); t2.start(); t1.join(); t2.join()
    return results

print("=== ไม่มี retry ===")
for k, v in run_two_transfers(use_retry=False).items():
    print(f"{k}: {v}")
display(query_df("SELECT user_id, followers FROM users WHERE user_id IN ('U001','U002')"))

print("=== มี retry (run_with_retry) ===")
for k, v in run_two_transfers(use_retry=True).items():
    print(f"{k}: {v}")
display(query_df("SELECT user_id, followers FROM users WHERE user_id IN ('U001','U002')"))
print("เมื่อ retry ครบ ทั้งสองโอน 10 สลับกัน ยอดรวมจึงกลับมาเท่าเดิม (U001=1500, U002=300)")

=== ไม่มี retry ===
T2 U002->U001: error 1213: Deadlock found when trying to get lock; try restarting transaction
T1 U001->U002: committed


,user_id,followers
0,U001,1490
1,U002,310


=== มี retry (run_with_retry) ===
  ↻ errno 1213 (attempt 1) -> retry อีกใน 0.11s
T2 U002->U001: committed
T1 U001->U002: committed


,user_id,followers
0,U001,1500
1,U002,300


เมื่อ retry ครบ ทั้งสองโอน 10 สลับกัน ยอดรวมจึงกลับมาเท่าเดิม (U001=1500, U002=300)


### 2.3.4 ทดสอบ Isolation Level อย่างปลอดภัย (งาน C)
ใช้ 2 threads + `Event` ประสานลำดับ: reader อ่านครั้งแรก → writer UPDATE เป็น 999 + COMMIT → reader อ่านครั้งที่สอง
writer ตั้ง `innodb_lock_wait_timeout=3` เพื่อไม่ให้ค้างนานกรณี SERIALIZABLE

In [ ]:
def reader(level, ready, writer_done, result):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute(f"SET SESSION TRANSACTION ISOLATION LEVEL {level}")
        cur.execute("START TRANSACTION")
        cur.execute("SELECT share_count FROM posts WHERE post_id='P001'")
        result['first'] = cur.fetchone()[0]
        ready.set()
        writer_done.wait(timeout=8)
        cur.execute("SELECT share_count FROM posts WHERE post_id='P001'")
        result['second'] = cur.fetchone()[0]
        conn.commit()
    finally:
        cur.close(); conn.close()

def writer(ready, writer_done, result):
    ready.wait(timeout=8)
    conn = get_connection(); cur = conn.cursor()
    t0 = time.time()
    try:
        cur.execute("SET SESSION innodb_lock_wait_timeout=3")
        cur.execute("UPDATE posts SET share_count=999 WHERE post_id='P001'")
        conn.commit(); result['writer'] = 'committed'
    except Error as e:
        conn.rollback(); result['writer'] = f"error {e.errno}"
    finally:
        result['writer_wait_s'] = round(time.time() - t0, 2)
        cur.close(); conn.close(); writer_done.set()

def run_isolation_test(level):
    ready = threading.Event(); done = threading.Event(); result = {}
    t1 = threading.Thread(target=reader, args=(level, ready, done, result))
    t2 = threading.Thread(target=writer, args=(ready, done, result))
    t1.start(); t2.start(); t1.join(); t2.join()
    return result

rows = []
for level in ['READ COMMITTED', 'REPEATABLE READ', 'SERIALIZABLE']:
    reset_share()   # เริ่มที่ 0 ทุกรอบ
    r = run_isolation_test(level)
    rows.append({'isolation': level, 'first read': r.get('first'), 'second read': r.get('second'),
                 'writer': r.get('writer'), 'writer รอ (s)': r.get('writer_wait_s'),
                 'ค่าสุดท้ายใน DB': get_share()})
iso_df = pd.DataFrame(rows)
iso_df

,isolation,first read,second read,writer,writer รอ (s),ค่าสุดท้ายใน DB
0,READ COMMITTED,0,999,committed,0.01,999
1,REPEATABLE READ,0,0,committed,0.02,999
2,SERIALIZABLE,0,0,error 1205,3.00,0


**แนวอภิปรายผล**
* **READ COMMITTED** — second read = 999 เพราะ consistent read แต่ละ statement ใช้ snapshot ใหม่ → *non-repeatable read*
* **REPEATABLE READ** — second read ยังเป็น 0 เพราะใช้ snapshot ที่สร้างตอน consistent read ครั้งแรก writer ไม่ถูก block (MVCC)
* **SERIALIZABLE** — plain SELECT (autocommit ปิด) กลายเป็น shared locking read writer ต้องรอ S lock ของ reader จนได้ error 1205 (lock wait timeout) → แลก concurrency กับ isolation

## 2.4 ตรวจสอบ Redo Log และ Recovery

### 2.4.1 ดู InnoDB Status
ผลลัพธ์ยาวมาก จึงตัดมาเฉพาะส่วนที่เกี่ยวกับแลป: `LOG` (LSN / checkpoint), `TRANSACTIONS` และ `LATEST DETECTED DEADLOCK` (ถ้ามี)

In [ ]:
import re

def innodb_status():
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SHOW ENGINE INNODB STATUS")
    status = cur.fetchone()[2]
    cur.close(); conn.close()
    return status

def status_section(status, name):
    # ดึงข้อความของหัวข้อ เช่น 'LOG' จาก SHOW ENGINE INNODB STATUS
    m = re.search(rf"-+\n{re.escape(name)}\n-+\n(.*?)(?=\n-+\n[A-Z][A-Z /]+\n-+\n|\Z)", status, re.S)
    return m.group(1).strip() if m else f"(ไม่พบหัวข้อ {name})"

status = innodb_status()
print("=== LOG ===\n" + status_section(status, 'LOG'))
print("\n=== TRANSACTIONS (ย่อ) ===\n" + status_section(status, 'TRANSACTIONS')[:800])
# print(status)   # เอา comment ออกถ้าต้องการดูทั้งหมด

=== LOG ===
Log capacity                 104857600
Log capacity used            104857600
Log sequence number          19586578
Log buffer assigned up to    19586578
Log buffer completed up to   19586578
Log written up to            19586578
Log flushed up to            19586578
Added dirty pages up to      19586578
Pages flushed up to          19586073
Last checkpoint at           19586073
Log minimum file id is       5
Log maximum file id is       5
315 log i/o's done, 16.58 log i/o's/second

=== TRANSACTIONS (ย่อ) ===
Trx id counter 2452
Purge done for trx's n:o < 2451 undo n:o < 0 state: running but idle
History list length 5
LIST OF TRANSACTIONS FOR EACH SESSION:
---TRANSACTION 421795701419224, not started
0 lock struct(s), heap size 1128, 0 row lock(s)
---TRANSACTION 421795701418416, not started
0 lock struct(s), heap size 1128, 0 row lock(s)
---TRANSACTION 421795701417608, not started
0 lock struct(s), heap size 1128, 0 row lock(s)


### 2.4.2 ดู Redo Log Configuration ที่เป็นปัจจุบัน
ใน MySQL 8.4 ใช้ `innodb_redo_log_capacity` (redo files อยู่ในไดเรกทอรี `#innodb_redo`) แทน `innodb_log_file_size` / `innodb_log_files_in_group`

In [ ]:
redo_vars = pd.concat([
    query_df("SHOW VARIABLES LIKE 'innodb_redo_log_capacity'"),
    query_df("SHOW GLOBAL STATUS LIKE 'Innodb_redo_log_capacity_resized'"),
    query_df("SHOW GLOBAL STATUS LIKE 'Innodb_os_log_written'"),
    query_df("SHOW VARIABLES LIKE 'innodb_log_buffer_size'"),
    query_df("SHOW VARIABLES LIKE 'innodb_flush_log_at_trx_commit'"),
    query_df("SHOW VARIABLES LIKE 'sync_binlog'"),
    query_df("SHOW VARIABLES LIKE 'innodb_deadlock_detect'"),
    query_df("SHOW VARIABLES LIKE 'innodb_lock_wait_timeout'"),
], ignore_index=True)

def human_bytes(v):
    try:
        v = int(v)
    except (TypeError, ValueError):
        return ''
    for unit in ['B', 'KB', 'MB', 'GB']:
        if v < 1024: return f"{v:.0f} {unit}"
        v /= 1024
    return f"{v:.1f} TB"

redo_vars['อ่านง่าย'] = [human_bytes(v) if 'size' in n or 'capacity' in n.lower() or 'written' in n else ''
                        for n, v in zip(redo_vars.iloc[:, 0], redo_vars.iloc[:, 1])]
redo_vars

,Variable_name,Value,อ่านง่าย
0,innodb_redo_log_capacity,104857600,100 MB
1,Innodb_redo_log_capacity_resized,104857600,100 MB
2,Innodb_os_log_written,346112,338 KB
3,innodb_log_buffer_size,16777216,16 MB
4,innodb_flush_log_at_trx_commit,1,
5,sync_binlog,1,
6,innodb_deadlock_detect,ON,
7,innodb_lock_wait_timeout,50,


### 2.4.3 สาธิต Redo Generation (งาน E)
เก็บ counter `Innodb_os_log_written` ก่อนและหลัง workload เป็นค่าประมาณ เพราะเป็น global counter ที่ workload อื่นบน server ก็ส่งผลได้

In [ ]:
def get_redo_bytes():
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SHOW GLOBAL STATUS LIKE 'Innodb_os_log_written'")
    value = int(cur.fetchone()[1])
    cur.close(); conn.close()
    return value

def get_lsn():
    m = re.search(r"Log sequence number\s+(\d+)", innodb_status())
    return int(m.group(1)) if m else None

N_ROWS = 1000
before, lsn_before = get_redo_bytes(), get_lsn()
conn = get_connection(); cur = conn.cursor()
try:
    cur.execute("DELETE FROM posts WHERE post_id LIKE 'BULK\\_%'"); conn.commit()
    before, lsn_before = get_redo_bytes(), get_lsn()   # วัดหลังเคลียร์ข้อมูลเก่า
    for i in range(N_ROWS):
        cur.execute('''
            INSERT INTO posts(post_id, author_id, content, share_count)
            VALUES (%s, 'U001', %s, 0)
        ''', (f'BULK_{i:04d}', f'Bulk insert test {i}'))
    conn.commit()
finally:
    cur.close(); conn.close()
after, lsn_after = get_redo_bytes(), get_lsn()

print(f"Redo bytes generated (approx.): {after - before:,} bytes  (~{(after - before) / N_ROWS:.0f} bytes/แถว)")
if lsn_before and lsn_after:
    print(f"LSN: {lsn_before:,} -> {lsn_after:,}  (เพิ่ม {lsn_after - lsn_before:,})")

Redo bytes generated (approx.): 376,832 bytes  (~377 bytes/แถว)
LSN: 19,586,588 -> 19,740,033  (เพิ่ม 153,445)


In [ ]:
# ทำความสะอาดแถว BULK (การลบก็สร้าง redo + undo เช่นกัน)
b = get_redo_bytes()
with get_connection() as conn:
    cur = conn.cursor()
    cur.execute("DELETE FROM posts WHERE post_id LIKE 'BULK\\_%'")
    print("ลบแถว BULK:", cur.rowcount)
    conn.commit(); cur.close()
print(f"redo bytes จากการ DELETE ≈ {get_redo_bytes() - b:,}")

ลบแถว BULK: 1000
redo bytes จากการ DELETE ≈ 224,768


## 2.5 แบบฝึกหัดปฏิบัติ (Lab Exercises)

### Lab 1: วิเคราะห์ Dirty Read
Connection 1 UPDATE เป็น 777 แต่ยังไม่ COMMIT → Connection 2 อ่านภายใต้ isolation ต่าง ๆ → Connection 1 ROLLBACK
(plain SELECT ไม่ถูก block ทั้งสองระดับ จึงรันตามลำดับใน thread เดียวได้)

In [ ]:
def dirty_read_test(level):
    reset_share()
    c1 = get_connection(); cur1 = c1.cursor()
    c2 = get_connection(); cur2 = c2.cursor()
    try:
        cur1.execute("START TRANSACTION")
        cur1.execute("UPDATE posts SET share_count=777 WHERE post_id='P001'")   # no commit

        cur2.execute(f"SET SESSION TRANSACTION ISOLATION LEVEL {level}")
        cur2.execute("SELECT share_count FROM posts WHERE post_id='P001'")
        seen_during = cur2.fetchone()[0]
        c2.commit()

        c1.rollback()   # T1 ยกเลิก
        cur2.execute("SELECT share_count FROM posts WHERE post_id='P001'")
        seen_after = cur2.fetchone()[0]
        c2.commit()
        return {'isolation (T2)': level, 'T2 เห็นระหว่าง T1 ยังไม่ commit': seen_during,
                'ค่าหลัง T1 rollback': seen_after, 'dirty read?': seen_during == 777}
    finally:
        cur1.close(); c1.close(); cur2.close(); c2.close()

pd.DataFrame([dirty_read_test(l) for l in ['READ UNCOMMITTED', 'READ COMMITTED']])

,isolation (T2),T2 เห็นระหว่าง T1 ยังไม่ commit,ค่าหลัง T1 rollback,dirty read?
0,READ UNCOMMITTED,777,0,True
1,READ COMMITTED,0,0,False


**สรุป:** ภายใต้ READ UNCOMMITTED, T2 เห็นค่า 777 ซึ่งไม่เคยมีอยู่จริงหลัง T1 rollback ถ้า business logic ตัดสินใจจากค่านี้ (เช่น ขึ้น “ไวรัล” หรือคิดค่าโฆษณา) จะผิดพลาด READ COMMITTED ป้องกันได้เพราะอ่านเฉพาะเวอร์ชันที่ commit แล้ว (MVCC)

### Lab 2: Atomic Transaction สำหรับ Report Fake News
1. INSERT ลง `reports` โดยมี `UNIQUE(post_id, user_id)` ป้องกัน report ซ้ำ
2. `UPDATE posts.report_count = report_count + 1` (atomic)
3. ถ้าถึง threshold → `is_flagged = TRUE`
4. duplicate (1062) หรือ foreign key error (1452) → rollback ทั้งชุด

In [ ]:
FLAG_THRESHOLD = 3

def setup_reports_table():
    with get_connection() as conn:
        cur = conn.cursor()
        cur.execute('''
            CREATE TABLE IF NOT EXISTS reports (
                report_id   BIGINT AUTO_INCREMENT PRIMARY KEY,
                post_id     VARCHAR(20) NOT NULL,
                user_id     VARCHAR(20) NOT NULL,
                reason      VARCHAR(255) NOT NULL,
                reported_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                CONSTRAINT fk_reports_post FOREIGN KEY (post_id) REFERENCES posts(post_id) ON DELETE CASCADE,
                CONSTRAINT fk_reports_user FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE,
                CONSTRAINT uq_report UNIQUE (post_id, user_id)
            ) ENGINE=InnoDB
        ''')
        conn.commit(); cur.close()

class ReportError(Exception):
    pass

def report_fake_news(user_id, post_id, reason, threshold=FLAG_THRESHOLD):
    conn = get_connection(); cur = conn.cursor()
    try:
        # 1) บันทึก report (UNIQUE/FK ตรวจที่ DBMS)
        cur.execute("INSERT INTO reports(post_id, user_id, reason) VALUES (%s, %s, %s)",
                    (post_id, user_id, reason))
        # 2) atomic increment — ถือ X lock ที่ row ของโพสต์จน commit
        cur.execute("UPDATE posts SET report_count = report_count + 1 WHERE post_id = %s", (post_id,))
        if cur.rowcount != 1:
            raise ReportError(f"ไม่พบโพสต์ {post_id}")
        # 3) flag เมื่อถึง threshold (อ่านค่าหลังจากเราถือ lock อยู่แล้ว จึงไม่มี race)
        cur.execute('''
            UPDATE posts SET is_flagged = TRUE
            WHERE post_id = %s AND report_count >= %s AND is_flagged = FALSE
        ''', (post_id, threshold))
        flagged_now = cur.rowcount == 1
        conn.commit()
        return 'flagged' if flagged_now else 'reported'
    except Error as e:
        conn.rollback()
        if e.errno == 1062:
            raise ReportError(f"{user_id} รายงาน {post_id} ซ้ำ (duplicate) -> rollback") from e
        if e.errno == 1452:
            raise ReportError(f"post/user ไม่มีอยู่จริง (foreign key) -> rollback") from e
        raise
    except Exception:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

setup_reports_table()
insert_sample_data()

def try_report(u, p, reason='misleading'):
    try:
        return f"{u} -> {p}: {run_with_retry(lambda: report_fake_news(u, p, reason), verbose=False)}"
    except ReportError as e:
        return f"{u} -> {p}: ❌ {e}"

print("--- ทดสอบทีละขั้น ---")
for u, p in [('U001', 'P003'), ('U001', 'P003'), ('U002', 'PXXX'), ('U999', 'P003')]:
    print(try_report(u, p))
display(query_df("SELECT post_id, report_count, is_flagged FROM posts WHERE post_id='P003'"))
display(query_df("SELECT post_id, user_id, reason FROM reports"))

--- ทดสอบทีละขั้น ---
U001 -> P003: reported
U001 -> P003: ❌ U001 รายงาน P003 ซ้ำ (duplicate) -> rollback
U002 -> PXXX: ❌ post/user ไม่มีอยู่จริง (foreign key) -> rollback
U999 -> P003: ❌ post/user ไม่มีอยู่จริง (foreign key) -> rollback


,post_id,report_count,is_flagged
0,P003,1,0


,post_id,user_id,reason
0,P003,U001,misleading


In [ ]:
# --- ทดสอบ concurrent reports: 4 users รายงาน P002 พร้อมกัน + มี duplicate ปน ---
insert_sample_data()
jobs = [('U001', 'P002'), ('U002', 'P002'), ('U003', 'P002'), ('U004', 'P002'),
        ('U001', 'P002'), ('U003', 'P002')]   # 2 รายการสุดท้ายซ้ำ
with concurrent.futures.ThreadPoolExecutor(max_workers=len(jobs)) as ex:
    for line in ex.map(lambda j: try_report(*j), jobs):
        print(line)

check = query_df('''
    SELECT p.post_id, p.report_count, p.is_flagged,
           (SELECT COUNT(*) FROM reports r WHERE r.post_id = p.post_id) AS rows_in_reports
    FROM posts p WHERE p.post_id = 'P002'
''')
display(check)
ok = int(check.report_count[0]) == int(check.rows_in_reports[0]) == 4 and bool(check.is_flagged[0])
print("✅ report_count ตรงกับจำนวน report จริง และถูก flag" if ok else "❌ ไม่ตรงกัน ตรวจสอบโค้ด")

U001 -> P002: reported
U002 -> P002: reported
U003 -> P002: flagged
U004 -> P002: reported
U001 -> P002: ❌ U001 รายงาน P002 ซ้ำ (duplicate) -> rollback
U003 -> P002: ❌ U003 รายงาน P002 ซ้ำ (duplicate) -> rollback


,post_id,report_count,is_flagged,rows_in_reports
0,P002,4,1,4


✅ report_count ตรงกับจำนวน report จริง และถูก flag


**ทำไมถูกต้องภายใต้ concurrency:** `report_count = report_count + 1` เป็น atomic UPDATE (ไม่อ่านค่ามาบวกใน Python) และ X lock บน row ของโพสต์ถูกถือจน COMMIT ทำให้ statement ตรวจ threshold เห็นค่าล่าสุดเสมอ ส่วน report ซ้ำถูก `UNIQUE(post_id, user_id)` ตัดทิ้งแล้ว rollback ทั้งชุด จึงไม่มี `report_count` เกิน
(ถ้าเกิด deadlock ระหว่าง insert/update พร้อมกัน `run_with_retry` จะ retry ทั้ง transaction ให้)

### Lab 3: Benchmark Autocommit vs Explicit Transaction
* **Case A** — `autocommit=True` INSERT ทีละ statement (commit ทุกแถว → flush redo ทุกครั้งเมื่อ `innodb_flush_log_at_trx_commit=1`)
* **Case B** — `autocommit=False` INSERT ทั้งหมดแล้ว commit ครั้งเดียว

มี warm-up 1 รอบ และวัด 5 รอบ รายงาน median/mean

In [ ]:
import statistics

N_INSERT = 500
ROUNDS = 5

with get_connection() as conn:
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS bench_inserts (
            id INT AUTO_INCREMENT PRIMARY KEY,
            payload VARCHAR(100) NOT NULL
        ) ENGINE=InnoDB
    ''')
    conn.commit(); cur.close()

def truncate_bench():
    with get_connection() as conn:
        cur = conn.cursor(); cur.execute("TRUNCATE TABLE bench_inserts"); cur.close()

def case_a_autocommit(n):
    conn = get_connection(); conn.autocommit = True; cur = conn.cursor()
    try:
        t0 = time.perf_counter()
        for i in range(n):
            cur.execute("INSERT INTO bench_inserts(payload) VALUES (%s)", (f"row {i}",))
        return time.perf_counter() - t0
    finally:
        cur.close(); conn.close()

def case_b_explicit(n):
    conn = get_connection(); cur = conn.cursor()
    try:
        t0 = time.perf_counter()
        conn.start_transaction()
        for i in range(n):
            cur.execute("INSERT INTO bench_inserts(payload) VALUES (%s)", (f"row {i}",))
        conn.commit()
        return time.perf_counter() - t0
    except Error:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

results = {'A: autocommit ทีละแถว': [], 'B: explicit transaction เดียว': []}
for r in range(ROUNDS + 1):            # รอบ 0 = warm-up ไม่นับ
    for name, fn in [('A: autocommit ทีละแถว', case_a_autocommit),
                     ('B: explicit transaction เดียว', case_b_explicit)]:
        truncate_bench()
        rb = get_redo_bytes()
        t = fn(N_INSERT)
        if r > 0:
            results[name].append((t, get_redo_bytes() - rb))

bench = pd.DataFrame([{
    'case': name,
    'median (s)': round(statistics.median(x[0] for x in v), 4),
    'mean (s)': round(statistics.mean(x[0] for x in v), 4),
    'rows/s (median)': round(N_INSERT / statistics.median(x[0] for x in v)),
    'redo bytes (median)': int(statistics.median(x[1] for x in v)),
} for name, v in results.items()])
display(bench)
speedup = bench['median (s)'][0] / bench['median (s)'][1]
print(f"Explicit transaction เร็วกว่า ~{speedup:.1f} เท่า "
      f"(innodb_flush_log_at_trx_commit = {query_df('SELECT @@innodb_flush_log_at_trx_commit').iloc[0,0]})")
truncate_bench()

,case,median (s),mean (s),rows/s (median),redo bytes (median)
0,A: autocommit ทีละแถว,4.4055,4.2545,113,518656
1,B: explicit transaction เดียว,0.2127,0.2123,2351,132608


Explicit transaction เร็วกว่า ~20.7 เท่า (innodb_flush_log_at_trx_commit = 1)


**อภิปราย:** commit บ่อยทำให้มี log flush/fsync และ round-trip ของ COMMIT ทุกแถว ส่วน transaction เดียวจ่ายค่า flush ครั้งเดียว แต่ transaction ที่ใหญ่เกินไปก็มีต้นทุน — undo log สะสม, ถือ lock นาน, rollback/recovery นานขึ้น แนวปฏิบัติจึงมักแบ่งเป็น batch ขนาดพอเหมาะ (เช่น 500–5,000 แถวต่อ commit)

### Lab 4: Deadlock Detection และ Retry
สร้าง deadlock ด้วย UPDATE 2 แถวในลำดับตรงข้ามกัน (ไม่ใช้ `LOCK TABLES`) ตาม script ของเอกสาร:

```
T1: UPDATE ... user_id='U001';   -- wait
T2: UPDATE ... user_id='U002';   -- wait
T1: UPDATE ... user_id='U002';   -- รอ T2
T2: UPDATE ... user_id='U001';   -- วงจร -> deadlock
```

In [ ]:
def follow_both(first, second, step_barrier=None, log=None, name=''):
    conn = get_connection(); cur = conn.cursor()
    try:
        conn.start_transaction()
        cur.execute("UPDATE users SET followers=followers+1 WHERE user_id=%s", (first,))
        if log is not None: log.append(f"{name}: lock {first}")
        if step_barrier is not None:
            try: step_barrier.wait(timeout=5)
            except threading.BrokenBarrierError: pass
        cur.execute("UPDATE users SET followers=followers+1 WHERE user_id=%s", (second,))
        if log is not None: log.append(f"{name}: lock {second}")
        conn.commit()
        if log is not None: log.append(f"{name}: COMMIT")
        return 'committed'
    except Error as e:
        conn.rollback()
        if log is not None: log.append(f"{name}: error {e.errno} -> ROLLBACK")
        raise
    finally:
        cur.close(); conn.close()

def lab4(use_retry):
    insert_sample_data()
    barrier, log, out = threading.Barrier(2), [], {}
    def job(name, a, b):
        attempt = [0]
        def tx():
            attempt[0] += 1
            return follow_both(a, b, barrier if attempt[0] == 1 else None, log, name)
        try:
            out[name] = run_with_retry(tx, verbose=False) if use_retry else tx()
        except Error as e:
            out[name] = f"error {e.errno}"
        out[name] += f" (attempts={attempt[0]})"
    ts = [threading.Thread(target=job, args=('T1', 'U001', 'U002')),
          threading.Thread(target=job, args=('T2', 'U002', 'U001'))]
    for t in ts: t.start()
    for t in ts: t.join()
    return log, out

for use_retry in (False, True):
    log, out = lab4(use_retry)
    print(f"===== {'มี' if use_retry else 'ไม่มี'} retry =====")
    print("\n".join(log)); print(out)
    display(query_df("SELECT user_id, followers FROM users WHERE user_id IN ('U001','U002')"))
print("คาดหวังเมื่อ retry: U001 = 1502, U002 = 302 (ทั้งสอง transaction สำเร็จ)")

===== ไม่มี retry =====
T2: lock U002
T1: lock U001
T1: lock U002
T2: error 1213 -> ROLLBACK
T1: COMMIT
{'T2': 'error 1213 (attempts=1)', 'T1': 'committed (attempts=1)'}


,user_id,followers
0,U001,1501
1,U002,301


===== มี retry =====
T1: lock U001
T2: lock U002
T1: lock U002
T2: error 1213 -> ROLLBACK
T1: COMMIT
T2: lock U002
T2: lock U001
T2: COMMIT
{'T1': 'committed (attempts=1)', 'T2': 'committed (attempts=2)'}


,user_id,followers
0,U001,1502
1,U002,302


คาดหวังเมื่อ retry: U001 = 1502, U002 = 302 (ทั้งสอง transaction สำเร็จ)


In [ ]:
# ตรวจ deadlock ล่าสุดที่ InnoDB บันทึกไว้
dl = status_section(innodb_status(), 'LATEST DETECTED DEADLOCK')
print(dl[:2500])
print("\nจุดที่ควรสังเกต: '*** (1) TRANSACTION', '*** (2) HOLDS THE LOCK(S)', "
      "'*** (2) WAITING FOR THIS LOCK' และ '*** WE ROLL BACK TRANSACTION (x)' = victim")

2026-10-03 04:35:47 140320261109440
*** (1) TRANSACTION:
TRANSACTION 5777, ACTIVE 0 sec starting index read
mysql tables in use 1, locked 1
LOCK WAIT 3 lock struct(s), heap size 1128, 2 row lock(s), undo log entries 1
MySQL thread id 186, OS thread handle 140320209696448, query id 8308 localhost 127.0.0.1 lab updating
UPDATE users SET followers=followers+1 WHERE user_id='U002'

*** (1) HOLDS THE LOCK(S):
RECORD LOCKS space id 2 page no 4 n bits 72 index PRIMARY of table `fake_news_db`.`users` trx id 5777 lock_mode X locks rec but not gap
Record lock, heap no 2 PHYSICAL RECORD: n_fields 9; compact format; info bits 0
 0: len 4; hex 55303031; asc U001;;
 1: len 6; hex 000000001691; asc       ;;
 2: len 7; hex 010000015b0151; asc     [ Q;;
 3: len 5; hex 416c696365; asc Alice;;
 4: len 14; hex 616c69636540746573742e636f6d; asc alice@test.com;;
 5: len 4; hex 800005dd; asc     ;;
 6: len 4; hex 80000000; asc     ;;
 7: len 1; hex 81; asc  ;;
 8: len 4; hex 6ac08623; asc j  #;;


*** (1) WA

### ทำความสะอาด (ไม่บังคับ)

In [ ]:
DROP_ALL = False   # เปลี่ยนเป็น True ถ้าต้องการลบ database ของแลปทิ้ง
if DROP_ALL:
    with mysql.connector.connect(**SERVER_CONFIG) as conn:
        cur = conn.cursor(); cur.execute(f"DROP DATABASE IF EXISTS {DB_NAME}"); cur.close()
    print("ลบ", DB_NAME, "แล้ว")
else:
    insert_sample_data(); print("รีเซ็ตข้อมูลกลับเป็นค่าเริ่มต้นแล้ว")

รีเซ็ตข้อมูลกลับเป็นค่าเริ่มต้นแล้ว
